# Тестовое задание: RL для GridWorld

Все эксперименты здесь. Параметры сред и алгоритмов задаются прямо в ячейках.
Код среды и алгоритмов лежит в пакете `gridrl_v2` (`envs/`, `models/`, `utils.py`).

In [ ]:
import os
from pathlib import Path

# работаем из корня репозитория: туда пишутся runs/ и outputs/
if Path.cwd().name == "src_v2":
    os.chdir(Path.cwd().parent)

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, display

from gridrl_v2 import utils
from gridrl_v2.envs import make_env, optimal_stats
from gridrl_v2.models import DSAC, PPO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("учимся на", DEVICE)

## Задача 1. Среда

Четыре сетапа из задания: размер, число цветов пола (`None` — у каждой клетки свой цвет), доля препятствий.

In [ ]:
SETUPS = {
    1: dict(h=5, w=5, n_colors=None, obstacle_fraction=0.0),  # 25 уникальных цветов — полностью наблюдаемая (MDP)
    2: dict(h=5, w=5, n_colors=5, obstacle_fraction=0.0),
    3: dict(h=10, w=10, n_colors=7, obstacle_fraction=0.1),
    4: dict(h=10, w=10, n_colors=4, obstacle_fraction=0.1),
}

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))
for i in SETUPS:
    env = make_env(**SETUPS[i], seed=0)
    env.reset(seed=0)
    utils.draw_env(axes[i - 1], env)
    axes[i - 1].set_title(f"Env{i}: {SETUPS[i]}\nоптимальная длина пути {optimal_stats(env)['length']:.1f}", fontsize=9)
Path("outputs/envs").mkdir(parents=True, exist_ok=True)
fig.savefig("outputs/envs/setups.png", dpi=120, bbox_inches="tight")
plt.show()

## Задача 2. Базовые решения

Функции для графиков. По оси X — номер эпизода.
- `plot_metric` — для каждой среды отдельный график: синим полупрозрачным — значения по эпизодам, оранжевым пунктиром — скользящее среднее.
- `plot_compare` — скользящие средние всех сред на одном графике.

In [ ]:
def moving_average(values, window=50):
    # среднее по последним window эпизодам (nan пропускаем — у DSAC в начале лоссов нет)
    result = []
    for i in range(len(values)):
        last = [v for v in values[max(0, i - window + 1):i + 1] if not np.isnan(v)]
        if len(last) > 0:
            result.append(np.mean(last))
        else:
            result.append(np.nan)
    return result


def plot_metric(algs, key, title, name):
    fig, axes = plt.subplots(1, len(algs), figsize=(5 * len(algs), 3.5))
    for ax, (env_name, alg) in zip(axes, algs.items()):
        values = [row[key] for row in alg.history]
        ax.plot(values, color="tab:blue", alpha=0.3, label="по эпизодам")
        ax.plot(moving_average(values), color="tab:orange", linestyle="--", linewidth=2, label="среднее за 50")
        if key == "return":
            ax.axhline(alg.optimal["return"], color="green", linestyle=":", label="оптимум")
        if key == "length":
            ax.axhline(alg.optimal["length"], color="green", linestyle=":", label="оптимум")
            ax.set_yscale("log")
        ax.set_title(f"{env_name}: {title}")
        ax.set_xlabel("эпизод")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    fig.tight_layout()
    Path("outputs/plots").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"outputs/plots/{name}_{key}.png", dpi=110)
    plt.show()


def plot_compare(algs, key, title, name):
    plt.figure(figsize=(10, 5))
    for env_name, alg in algs.items():
        values = [row[key] for row in alg.history]
        plt.plot(moving_average(values), label=env_name)
    plt.title(title)
    plt.xlabel("эпизод")
    plt.ylabel(key)
    plt.grid(alpha=0.5)
    plt.legend()
    Path("outputs/plots").mkdir(parents=True, exist_ok=True)
    plt.savefig(f"outputs/plots/{name}_{key}_compare.png", dpi=110)
    plt.show()


def show_agent(alg, name, n_episodes=100):
    # как ходит обученный агент: тепловая карта посещений за 100 эпизодов и GIF первых 5
    episodes = alg.evaluate(n_episodes)
    success = np.mean([ep["success"] for ep in episodes])
    length = np.mean([ep["length"] for ep in episodes])
    title = f"{name}: success {success:.2f}, длина {length:.1f} (оптимум {alg.optimal['length']:.1f})"

    fig, ax = plt.subplots(figsize=(6, 6))
    image = utils.plot_heatmap(ax, alg.env, episodes)
    fig.colorbar(image, ax=ax, fraction=0.046, label="посещений за эпизод")
    ax.set_title(title, fontsize=9)
    Path("outputs/eval").mkdir(parents=True, exist_ok=True)
    fig.savefig(f"outputs/eval/{name}_heatmap.png", dpi=120, bbox_inches="tight")
    plt.show()

    # в GIF — до 5 коротких эпизодов (длинные дают тысячи кадров и рисуются очень долго)
    short = [ep for ep in episodes if ep["length"] <= 60][:5]
    if len(short) == 0:
        short = [episodes[0]]  # агент нигде не дошёл быстро — покажем один длинный эпизод
    utils.save_gif(alg.env, short, f"outputs/eval/{name}.gif", name)
    display(Image(filename=f"outputs/eval/{name}.gif"))

### PPO (on-policy)

Общие параметры — в `ppo_cfg`. Для каждой среды делаем копию и дописываем карту и число эпизодов:
`{**ppo_cfg, **SETUPS[i], ...}`.

In [ ]:
ppo_cfg = {
    "device": DEVICE,
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сеть и старты эпизодов
    "memory": True,  # True — GRU, False — MLP без памяти
    "episodes": 1000,
    "episodes_per_update": 10,  # сколько эпизодов собрать перед обновлением
    "epochs": 4,
    "episodes_per_batch": 5,
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "lr": 3e-4,
    "clip": 0.2,
    "ent_coef": 0.01,
    "vf_coef": 0.5,
    "hidden": 128,
    "run_name": "ppo",
}

PPO_EPISODES = {1: 1000, 2: 1000, 3: 1500, 4: 1500}  # на больших картах учимся дольше

In [ ]:
ppo_algs = {}
for i in SETUPS:
    print(f"===== Env{i} =====")
    alg = PPO({**ppo_cfg, **SETUPS[i], "episodes": PPO_EPISODES[i], "run_name": f"ppo_env{i}"})
    alg.train()
    alg.save()
    ppo_algs[f"Env{i}"] = alg

In [ ]:
plot_metric(ppo_algs, "return", "средняя отдача", "ppo")
plot_metric(ppo_algs, "length", "длина эпизода", "ppo")
plot_metric(ppo_algs, "loss", "полный лосс", "ppo")
plot_metric(ppo_algs, "policy_loss", "лосс актора (policy)", "ppo")
plot_metric(ppo_algs, "value_loss", "лосс критика (value)", "ppo")
plot_metric(ppo_algs, "entropy", "энтропия", "ppo")
plot_metric(ppo_algs, "explore_prob", "вероятность нежадного действия", "ppo")

In [ ]:
plot_compare(ppo_algs, "return", "PPO: средняя отдача", "ppo")
plot_compare(ppo_algs, "policy_loss", "PPO: Policy Loss", "ppo")

In [ ]:
for env_name, alg in ppo_algs.items():
    show_agent(alg, f"ppo_{env_name}")

### Discrete SAC (off-policy)

`soft_q=True` — мягкая Q (энтропия в цели критика), `soft_q=False` — обычная.

In [ ]:
dsac_cfg = {
    "device": DEVICE,
    "env_seed": 0,  # от него зависит карта
    "seed": 0,  # от него зависят сети, старты эпизодов и выборка из буфера
    "memory": True,  # True — GRU, False — MLP без памяти
    "soft_q": True,  # мягкая Q (энтропия в цели критика) или обычная
    "episodes": 500,
    "buffer_episodes": 5000,  # сколько последних эпизодов помнить
    "learning_starts": 20,  # сколько эпизодов только копим опыт, не учась
    "updates_per_episode": 4,  # сколько обновлений после каждого эпизода
    "batch_size": 8,  # сколько случайных эпизодов из буфера в одном обновлении
    "gamma": 0.99,
    "tau": 0.005,  # скорость, с которой target-критики догоняют обычных
    "lr": 3e-4,
    "alpha": 0.005,  # температура: вес энтропии
    "hidden": 128,
    "run_name": "dsac",
}

DSAC_EPISODES = {1: 500, 2: 500, 3: 800, 4: 800}

In [ ]:
dsac_algs = {}
for i in SETUPS:
    print(f"===== Env{i} =====")
    alg = DSAC({**dsac_cfg, **SETUPS[i], "episodes": DSAC_EPISODES[i], "run_name": f"dsac_env{i}"})
    alg.train()
    alg.save()
    dsac_algs[f"Env{i}"] = alg

In [ ]:
plot_metric(dsac_algs, "return", "средняя отдача", "dsac")
plot_metric(dsac_algs, "length", "длина эпизода", "dsac")
plot_metric(dsac_algs, "loss", "полный лосс", "dsac")
plot_metric(dsac_algs, "q_loss", "лосс критиков", "dsac")
plot_metric(dsac_algs, "actor_loss", "лосс актора", "dsac")
plot_metric(dsac_algs, "entropy", "энтропия", "dsac")
plot_metric(dsac_algs, "explore_prob", "вероятность нежадного действия", "dsac")

In [ ]:
plot_compare(dsac_algs, "return", "DSAC: средняя отдача", "dsac")

In [ ]:
for env_name, alg in dsac_algs.items():
    show_agent(alg, f"dsac_{env_name}")